# Sub-question 1: exposed working time

> In frontline community service occupations, how much working time is exposed
> to a language model alone, and how much is exposed only if the sector had
> software built for it?

Two published sources, joined on the O*NET task identifier:

- **hours per task per day** — Hatgis-Kessell et al. (2026)
- **exposure label E0 / E1 / E2** — Eloundou et al. (2023)

Eloundou's rubric distinguishes the two tiers the sub-question asks about:

> **Direct exposure (E1)**: "using the described LLM via ChatGPT or the OpenAI
> playground can decrease the time required to complete the DWA or task by at
> least half (50%)."
>
> **LLM+ Exposed (E2)**: "access to the described LLM alone would **not** reduce
> the time required to complete the activity/task by at least half, but
> additional software could be developed on top of the LLM that could."

**Exposure is a rated reduction in completion time, not removal of the task.**
A task rated E1 that takes an hour is rated as one that could take half an
hour — not zero.

Nothing in this notebook is aggregated, binned, reweighted or categorised.
Each figure shows one occupation's tasks as published.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## The three occupations

In [ ]:
# ---- the three occupations -------------------------------------------------
# OEWS May 2025 industry file, NAICS 624200. The three largest occupations in
# the industry by employment: two frontline service-delivery roles and the
# supervisory role above them.
STUDY = ["21-1093",   # Social and Human Service Assistants
         "21-1021",   # Child, Family, and School Social Workers
         "11-9151"]   # Social and Community Service Managers

oews = pd.read_excel("data/raw/oews/oesm25in4/oesm25in4/nat4d_M2025_dl.xlsx",
                     usecols=["NAICS", "OCC_CODE", "OCC_TITLE", "O_GROUP",
                              "TOT_EMP", "PCT_TOTAL"])
oews = oews[oews.NAICS.astype(str) == "624200"]
occ = oews[oews.O_GROUP == "detailed"].copy()
for c in ["TOT_EMP", "PCT_TOTAL"]:
    occ[c] = pd.to_numeric(occ[c], errors="coerce")

INDUSTRY_EMP = float(oews.loc[oews.O_GROUP == "total", "TOT_EMP"].iloc[0])
study = occ[occ.OCC_CODE.isin(STUDY)].set_index("OCC_CODE").loc[STUDY].reset_index()

print(f"NAICS 624200: {len(occ)} detailed occupations, {INDUSTRY_EMP:,.0f} jobs")
print(f"these three:  {study.TOT_EMP.sum():,.0f} jobs "
      f"({100 * study.TOT_EMP.sum() / INDUSTRY_EMP:.1f}% of the industry)")
study[["OCC_CODE", "OCC_TITLE", "TOT_EMP", "PCT_TOTAL"]]

## Hours and exposure, per task

In [ ]:
# ---- hours and exposure label, per task ------------------------------------
# hours : estimated hours per day on the task        Hatgis-Kessell et al. (2026)
# E     : E0 / E1 / E2                               Eloundou et al. (2023)
#
#   E1  a language model alone could cut the task's completion time by >= half
#   E2  the model alone could not, but software built on top of it could
#   E0  neither
#
# Eloundou publish three rating passes. The human annotators' labels are used
# for all results here; the two GPT-4 passes are carried through for the
# robustness check. Note their published `beta` column is the GPT-4 rating
# recoded (E1 -> 1, E2 -> 0.5, E0 -> 0), not the human one.
hours = pd.read_excel("data/raw/time_share/task_time_share_estimates.xlsx")
hours.columns = ["task_id", "task", "onet_code", "occ_name",
                 "hours", "hours_per_instance", "instances_per_day"]
hours["OCC_CODE"] = hours.onet_code.str.split(".").str[0]

elo = pd.read_csv("data/raw/gpts_are_gpts/full_labelset.tsv", sep="\t",
                  usecols=["Task ID", "Task Type", "human_exposure_agg",
                           "gpt4_exposure", "gpt4_exposure_alt_rubric",
                           "beta"])
assert (elo.beta == elo.gpt4_exposure.map({"E0": 0.0, "E2": 0.5, "E1": 1.0})).all()

three = (hours[hours.OCC_CODE.isin(study.OCC_CODE)]
       .merge(elo, left_on="task_id", right_on="Task ID", how="left")
       .rename(columns={"human_exposure_agg": "E"}))   # primary

print(f"{len(three)} tasks, {three.OCC_CODE.nunique()} occupations, "
      f"{three.E.notna().sum()} with an exposure label")
print(f"hours per occupation: "
      f"{three.groupby('OCC_CODE').hours.sum().round(2).unique()}")
three[["OCC_CODE", "task", "hours", "E", "gpt4_exposure"]].head()

## One figure per occupation

In [ ]:
# ---- one figure per occupation ---------------------------------------------
# Every task as published: bar length is the estimated time per day, colour is
# the exposure label. Nothing is summed, binned or reweighted. Hours are shown
# in minutes (the published value x 60) because most tasks here take well under
# an hour; the underlying column and the tables below remain in hours.
#
# Sized for a journal page: 6.5 inches wide, so the figure is reproduced at or
# near 1:1 and no type falls below 8 pt. Task statements are O*NET's own
# wording, wrapped to at most two lines. Statements too long for two lines are
# cut at a word boundary on the second, marked with an ellipsis; the task
# tables below carry the full text.
import re
import textwrap

E_COLOR = {"E0": "#d9dade", "E2": "#4e9e6a", "E1": "#1b4d2a"}
E_LABEL = {"E0": "Not exposed",
           "E2": "Exposed with additional software",
           "E1": "Directly exposed"}
XMAX = 180            # minutes, shared across all three figures
WRAP = 52             # characters per label line, max two lines
PT = 8                # every text element in the figure
FIG_W = 6.5           # inches, a journal text column
GUTTER = 0.12         # inches between the labels and the axis


def label(task, lines=2):
    """Wrap to at most `lines`, truncating the last line if it overflows.

    Wrapping first and cutting second means both lines use the full column
    width; cutting the raw string first leaves the second line short.
    """
    s = re.sub(r"\s+", " ", task).rstrip(". ")
    wrapped = textwrap.wrap(s, WRAP)
    if len(wrapped) <= lines:
        return chr(10).join(wrapped)
    kept = wrapped[:lines]
    last = kept[-1]
    # make room for the ellipsis on the final line
    if len(last) > WRAP - 3:
        last = re.sub(r"\s+\S*$", "", last[:WRAP - 2])
    kept[-1] = last.rstrip(" ,;") + "..."
    return chr(10).join(kept)


for _, o in study.iterrows():
    t = three[three.OCC_CODE == o.OCC_CODE].sort_values("hours")
    wrapped = [label(s) for s in t.task]
    rows = sum(w.count(chr(10)) + 1 for w in wrapped)

    # the label column is sized from the widest rendered label, so text can
    # never run into the bars however long the statements are
    fig = plt.figure(figsize=(FIG_W, 0.155 * rows + 1.6))
    probe = fig.add_axes([0, 0, 1, 1])
    probe.set_yticks(range(len(t)))
    probe.set_yticklabels(wrapped, fontsize=PT, linespacing=1.2)
    fig.canvas.draw()
    label_w = max(x.get_window_extent().width
                  for x in probe.get_yticklabels()) / fig.dpi
    probe.remove()

    left = (label_w + GUTTER) / FIG_W
    ax = fig.add_axes([left, 0.10, 0.95 - left, 0.82])
    minutes = t.hours * 60
    ax.barh(range(len(t)), minutes, height=0.68,
            color=[E_COLOR[e] for e in t.E])
    for y, v in enumerate(minutes):
        ax.text(v + 2.4, y, f"{v:.0f}", va="center", fontsize=PT, color="#555")

    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(wrapped, fontsize=PT, linespacing=1.2, ha="left")
    ax.tick_params(axis="y", pad=label_w * 72 + GUTTER * 72)
    ax.set_ylim(-0.7, len(t) - 0.3)
    ax.set_xlim(0, XMAX)
    ax.set_xlabel("Estimated minutes per day", fontsize=PT)
    ax.tick_params(axis="x", labelsize=PT)
    ax.set_title(o.OCC_TITLE, loc="left", fontsize=PT + 1.5, pad=8)

    handles = [plt.Rectangle((0, 0), 1, 1, color=E_COLOR[e])
               for e in ["E1", "E2", "E0"]]
    ax.legend(handles, [E_LABEL[e] for e in ["E1", "E2", "E0"]],
              title="Exposure", title_fontsize=PT, fontsize=PT,
              frameon=False, loc="lower right", borderpad=0.2,
              labelspacing=0.35)
    ax.grid(axis="x", alpha=0.25, linewidth=0.5)
    ax.set_axisbelow(True)
    for side in ["top", "right", "left"]:
        ax.spines[side].set_visible(False)
    ax.tick_params(axis="y", length=0)
    plt.show()

## Task-level tables

Full O*NET task statements, including the eleven the figures cut at 150
characters.

In [ ]:
# ---- task-level tables, one per occupation ---------------------------------
# Every task as published. Hours are estimated hours of a 7-hour day
# (Hatgis-Kessell et al. 2026); E is the human-rated exposure label
# (Eloundou et al. 2023).
TITLES = dict(zip(study.OCC_CODE, study.OCC_TITLE))

def task_table(occ_code):
    t = (three[three.OCC_CODE == occ_code]
         .assign(Task=lambda d: d.task.str.rstrip("."),
                 Hours=lambda d: d.hours.round(2))
         .rename(columns={"Task Type": "Task type"})
         .sort_values("Hours", ascending=False))
    return t[["Task", "Task type", "Hours", "E"]].reset_index(drop=True)


for code in study.OCC_CODE:
    t = three[three.OCC_CODE == code]
    by_tier = t.groupby("E").hours.sum().reindex(["E1", "E2", "E0"]).fillna(0)
    print(f"{TITLES[code]}  ({code})")
    print(f"   {len(t)} tasks, {t.hours.sum():.0f}-hour day   "
          f"E1 {by_tier.E1:.2f}h   E2 {by_tier.E2:.2f}h   E0 {by_tier.E0:.2f}h")
print()
table_21_1093 = task_table("21-1093")
table_21_1021 = task_table("21-1021")
table_11_9151 = task_table("11-9151")

In [ ]:
# Social and Human Service Assistants (21-1093)
table_21_1093

In [ ]:
# Child, Family, and School Social Workers (21-1021)
table_21_1021

In [ ]:
# Social and Community Service Managers (11-9151)
table_11_9151

## Robustness: human versus GPT-4 ratings

Eloundou et al. rated every task three times -- once by human annotators and
twice by GPT-4 under different rubrics. The results above use the human
ratings. The two passes agree on 60% of these 219 tasks, and the choice
changes the numbers materially, so both are reported.

In [ ]:
# ---- robustness: does the rater change the answer? -------------------------
# Eloundou et al. rated every task twice, once by human annotators and once by
# GPT-4. All results above use the human ratings. This cell reports the same
# hours under each, as published.
RATERS = [("Human-rated", "E"), ("GPT4-rated", "gpt4_exposure")]
TIERS = [("E1", "direct exposure (E1)"),
         ("E2", "partial exposure (E2)")]

rater = pd.DataFrame({
    f"{who} {name}": (three[three[col] == tier].groupby("OCC_CODE").hours.sum())
    for who, col in RATERS for tier, name in TIERS
}).reindex(study.OCC_CODE).fillna(0.0)
rater.index = [TITLES[c] for c in rater.index]

agree = (three.E == three.gpt4_exposure).mean()
print(f"human and GPT-4 labels agree on {agree:.0%} of the {len(three)} tasks")
print()
for c in rater.columns:
    print(f"  {c:<38} {rater[c].sum():5.2f}h")
rater.round(2)